# CarbonIQ: Electricity Consumption Forecasting & Carbon Estimation

**Project Type:** College Mini-Project

**Objective:** Build a reproducible ML pipeline that forecasts next-month household electricity consumption from two prior months of genuine consecutive observations, then estimates associated CO₂ emissions using a versioned, documented emission factor.

**Scope:**
- Dataset: Pune Household Survey (1,872 households, 26 monthly electricity columns spanning Jan 2018–Feb 2020)
- ML Task: Next-month electricity prediction (kWh) using t-2 and t-1 consumption + calendar month
- Train/Test Split: Chronological — train on data before Nov 2019, test on Nov 2019–Feb 2020
- Models Compared: Naive Persistence, Linear Regression, Random Forest, Gradient Boosting, Seasonal Gradient Boosting
- Carbon Calculation: Deterministic, using CEA v22.0 emission factor (0.675 kg CO₂/kWh, FY 2025-26 reporting period)
- Deliverables: Trained model artifact, metadata, FastAPI backend package with smoke tests

**Key Design Decisions:**
- Only *genuine consecutive 3-month windows* are used (no imputation across gaps)
- Extreme observations (e.g., 6,634 kWh) are preserved — they are real survey records
- The Seasonal Gradient Boosting model achieves the lowest MAE (32.12 kWh); standard Gradient Boosting has slightly better RMSE (185.60) and R² (0.176)
- Carbon calculation is separate from the ML model — the emission factor is configurable and versioned
- Forecasting accuracy is modest (R² ≈ 0.17); this is documented, not overstated

## B. Imports and Configuration

In [ ]:
import os
import json
import zipfile
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

# Reproducibility
RANDOM_STATE = 42

# Paths
DATA_PATH = "/content/Pune_HHLD_Survey_Data_v2-0-0.csv"
ARTIFACT_DIR = Path("/content/carboniq_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ARTIFACT_DIR / "carboniq_seasonal_gb.joblib"
METADATA_PATH = ARTIFACT_DIR / "carboniq_model_metadata.json"
BACKEND_ZIP_PATH = Path("/content/CarbonIQ_Backend_Package.zip")

# Versioned emission factor (source: Central Electricity Authority, database v22.0)
ELECTRICITY_FACTOR = {
    "kg_co2_per_kwh": 0.675,
    "source": "Central Electricity Authority (CEA)",
    "database_version": "22.0",
    "reporting_period": "FY 2025-26",
    "gas": "CO2"
}

print("Imports and configuration complete.")
print(f"Artifact directory: {ARTIFACT_DIR}")

## C. Dataset Loading and Essential Validation

In [ ]:
df = pd.read_csv(
    DATA_PATH,
    encoding="cp1252",
    low_memory=False
)

print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# Essential validation — only what informs downstream decisions
print("\nColumn dtypes:")
print(df.dtypes.value_counts())

# Check monthly electricity columns exist
monthly_cols = [
    col for col in df.columns
    if any(month in col for month in [
        "Jan_", "Feb_", "Mar_", "April_", "May_", "June_",
        "July_", "Aug_", "Sep_", "Oct_", "Nov_", "Dec_"
    ])
]
print(f"\nMonthly electricity columns found: {len(monthly_cols)}")

## D. Monthly Column Identification and Chronological Ordering

In [ ]:
month_order = {
    "Jan": 1, "Feb": 2, "Mar": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "Aug": 8,
    "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}

def month_key(col):
    parts = col.split("_")
    return (int(parts[1]), month_order[parts[0]])

monthly_cols_chrono = sorted(monthly_cols, key=month_key)

print("Monthly columns in chronological order:")
for col in monthly_cols_chrono:
    print(f"  {col}")

# Verify coverage
print("\nNon-null counts per month:")
print(df[monthly_cols_chrono].notna().sum())

## E. Construction of Genuine Consecutive-Month ML Windows

In [ ]:
# Build ML dataset: only genuine consecutive 3-month windows (t-2, t-1, target)
records = []

for household_idx, row in df[monthly_cols_chrono].iterrows():
    for i in range(2, len(monthly_cols_chrono)):
        prev_2 = row[monthly_cols_chrono[i - 2]]
        prev_1 = row[monthly_cols_chrono[i - 1]]
        target = row[monthly_cols_chrono[i]]

        if pd.notna(prev_2) and pd.notna(prev_1) and pd.notna(target):
            records.append({
                "household_id": household_idx,
                "month_t_minus_2": monthly_cols_chrono[i - 2],
                "month_t_minus_1": monthly_cols_chrono[i - 1],
                "target_month": monthly_cols_chrono[i],
                "electricity_t_minus_2": prev_2,
                "electricity_t_minus_1": prev_1,
                "electricity_target": target
            })

ml_df = pd.DataFrame(records)

print(f"ML dataset shape: {ml_df.shape}")
print(f"\nMissing values:\n{ml_df.isna().sum()}")
print(f"\nTarget statistics:")
print(ml_df["electricity_target"].describe())

print("\nFirst 5 records:")
display(ml_df.head())

## F. Minimal EDA and Feature Preparation

In [ ]:
# Target distribution percentiles (document extreme values without removing them)
print("Target percentiles:")
print(ml_df["electricity_target"].quantile(
    [0, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 1.0]
))

# Household-level features available for enrichment
household_features = ["adults", "children", "seniors", "rooms", "income"]

feature_check = pd.DataFrame({
    "Feature": household_features,
    "Missing": [df[col].isna().sum() for col in household_features],
    "Available": [df[col].notna().sum() for col in household_features]
})
print("\nHousehold feature availability:")
display(feature_check)

# Enrich ML dataset with household features + calendar month number
expanded_ml_df = ml_df.copy()

for feature in household_features:
    expanded_ml_df[feature] = expanded_ml_df["household_id"].map(df[feature])

expanded_ml_df["target_month_num"] = (
    expanded_ml_df["target_month"]
    .str.split("_")
    .str[0]
    .map(month_order)
)

print(f"\nExpanded ML dataset shape: {expanded_ml_df.shape}")
print(f"\nMissing values in added features:\n{expanded_ml_df[household_features + ['target_month_num']].isna().sum()}")

## G. Chronological Train/Test Split

In [ ]:
# Convert target month to sortable period: YYYYMM
target_year = (
    expanded_ml_df["target_month"]
    .str.split("_")
    .str[1]
    .astype(int)
)
expanded_ml_df["target_period"] = target_year * 100 + expanded_ml_df["target_month_num"]

# Train: before Nov 2019 (period < 201911)
# Test: Nov 2019 through Feb 2020 (201911–202002)
train_mask = expanded_ml_df["target_period"] < 201911
test_mask = expanded_ml_df["target_period"].between(201911, 202002)

train_data = expanded_ml_df[train_mask].copy()
test_data = expanded_ml_df[test_mask].copy()

print(f"Training samples: {len(train_data)}")
print(f"Testing samples: {len(test_data)}")

print("\nTraining period breakdown:")
print(train_data.groupby("target_month")["electricity_target"].count())

print("\nTesting period breakdown:")
print(test_data.groupby("target_month")["electricity_target"].count())

## H. Consistent Comparison of Naive, Linear Regression, Random Forest, Gradient Boosting, and Seasonal Gradient Boosting

In [ ]:
# Define feature sets
base_features = ["electricity_t_minus_2", "electricity_t_minus_1"]
seasonal_features = base_features + ["target_month_num"]

X_train_base = train_data[base_features]
X_test_base = test_data[base_features]
X_train_seasonal = train_data[seasonal_features]
X_test_seasonal = test_data[seasonal_features]

y_train = train_data["electricity_target"]
y_test = test_data["electricity_target"]

# Naive persistence baseline: predict t-1 as next month
naive_pred = test_data["electricity_t_minus_1"].to_numpy()

# Models to evaluate
models = {
    "Linear Regression": (
        LinearRegression(),
        X_train_base,
        X_test_base
    ),
    "Random Forest": (
        RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
        X_train_base,
        X_test_base
    ),
    "Gradient Boosting": (
        GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE),
        X_train_base,
        X_test_base
    ),
    "Seasonal Gradient Boosting": (
        GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE),
        X_train_seasonal,
        X_test_seasonal
    )
}

# Evaluate all models on the same test set
results = []

results.append({
    "Model": "Naive Baseline",
    "MAE": mean_absolute_error(y_test, naive_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, naive_pred)),
    "R2": r2_score(y_test, naive_pred)
})

for name, (model, X_train, X_test) in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })

comparison_df = pd.DataFrame(results).sort_values("MAE")
print("Model comparison on chronological test period (Nov 2019–Feb 2020):")
display(comparison_df.round(4))

In [ ]:
# Month-by-month comparison of the two strongest models
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Get predictions from the two best models
gb_model = models["Gradient Boosting"][0]
gb_pred = gb_model.predict(X_test_base)

seasonal_model = models["Seasonal Gradient Boosting"][0]
seasonal_pred = seasonal_model.predict(X_test_seasonal)

monthly_eval = test_data[["target_month", "electricity_target"]].copy()
monthly_eval["GB_Predicted"] = gb_pred
monthly_eval["Seasonal_Predicted"] = seasonal_pred

monthly_results = []
for month, group in monthly_eval.groupby("target_month"):
    actual = group["electricity_target"]
    for model_name, pred_col in [
        ("Gradient Boosting", "GB_Predicted"),
        ("Seasonal Gradient Boosting", "Seasonal_Predicted")
    ]:
        predicted = group[pred_col]
        monthly_results.append({
            "Month": month,
            "Model": model_name,
            "Samples": len(group),
            "MAE": mean_absolute_error(actual, predicted),
            "RMSE": np.sqrt(mean_squared_error(actual, predicted))
        })

monthly_df = pd.DataFrame(monthly_results)
print("Month-by-month MAE/RMSE for top two models:")
display(monthly_df.round(2))

## I. Final Model Selection and Evaluation

In [ ]:
# Final model: Seasonal Gradient Boosting (lowest MAE)
final_features = seasonal_features
X_train_final = train_data[final_features]
y_train_final = train_data["electricity_target"]
X_test_final = test_data[final_features]
y_test_final = test_data["electricity_target"]

final_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=RANDOM_STATE
)

final_model.fit(X_train_final, y_train_final)

final_predictions = final_model.predict(X_test_final)

final_metrics = {
    "MAE_kWh": float(mean_absolute_error(y_test_final, final_predictions)),
    "RMSE_kWh": float(np.sqrt(mean_squared_error(y_test_final, final_predictions))),
    "R2": float(r2_score(y_test_final, final_predictions))
}

print("Final model: Seasonal Gradient Boosting")
print(f"Training samples: {len(X_train_final)}")
print(f"Testing samples: {len(X_test_final)}")
print("\nVerified evaluation metrics (computed from predictions):")
for metric, value in final_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Feature importance
importance_df = pd.DataFrame({
    "Feature": final_features,
    "Importance": final_model.feature_importances_
}).sort_values("Importance", ascending=False)
print("\nFeature importance:")
display(importance_df)

## J. Deterministic Electricity Carbon Calculator

In [ ]:
def calculate_electricity_co2(consumption_kwh):
    """
    Calculate grid-electricity emissions from monthly
    electricity consumption in kWh.

    Returns emissions in kg CO2 and tonnes CO2.
    """
    consumption = float(consumption_kwh)

    if not np.isfinite(consumption) or consumption < 0:
        raise ValueError(
            "Electricity consumption must be a finite, non-negative number."
        )

    factor = ELECTRICITY_FACTOR["kg_co2_per_kwh"]
    emissions_kg_co2 = consumption * factor

    return {
        "electricity_kwh": consumption,
        "emission_factor_kg_co2_per_kwh": factor,
        "emissions_kg_co2": round(emissions_kg_co2, 3),
        "emissions_t_co2": round(emissions_kg_co2 / 1000, 6),
        "factor_source": ELECTRICITY_FACTOR["source"],
        "factor_version": ELECTRICITY_FACTOR["database_version"],
        "reporting_period": ELECTRICITY_FACTOR["reporting_period"]
    }

# Test the calculator
example_result = calculate_electricity_co2(100)
display(pd.DataFrame([example_result]))

## K. Integrated Forecasting and Carbon Calculation Pipeline

In [ ]:
def forecast_next_month_co2(
    electricity_t_minus_2,
    electricity_t_minus_1,
    target_month_num
):
    """Predict next-month electricity consumption and estimate CO2 emissions."""

    # Validate historical consumption
    values = [electricity_t_minus_2, electricity_t_minus_1]

    if not all(np.isfinite(v) and v >= 0 for v in values):
        raise ValueError(
            "Previous electricity consumption must be finite and non-negative."
        )

    # Validate target month
    if (
        not isinstance(target_month_num, (int, np.integer))
        or not 1 <= target_month_num <= 12
    ):
        raise ValueError("Target month must be an integer from 1 to 12.")

    # Prepare model input in the exact feature order used during training
    model_input = pd.DataFrame([{
        "electricity_t_minus_2": electricity_t_minus_2,
        "electricity_t_minus_1": electricity_t_minus_1,
        "target_month_num": target_month_num
    }])

    # Predict electricity consumption
    raw_prediction = float(final_model.predict(model_input)[0])

    # Enforce physical constraint and round BEFORE calculating emissions
    predicted_kwh = round(max(0.0, raw_prediction), 2)

    # Calculate emissions using the same displayed consumption value
    carbon_result = calculate_electricity_co2(predicted_kwh)

    return {
        "previous_2_months_kwh": electricity_t_minus_2,
        "previous_month_kwh": electricity_t_minus_1,
        "target_month_num": target_month_num,
        "predicted_electricity_kwh": predicted_kwh,
        "predicted_co2_kg": carbon_result["emissions_kg_co2"],
        "emission_factor_kg_co2_per_kwh":
            carbon_result["emission_factor_kg_co2_per_kwh"],
        "factor_source": carbon_result["factor_source"],
        "factor_version": carbon_result["factor_version"],
        "reporting_period": carbon_result["reporting_period"]
    }

# Smoke-test the integrated pipeline
demo_result = forecast_next_month_co2(
    electricity_t_minus_2=92,
    electricity_t_minus_1=114,
    target_month_num=3
)

display(pd.DataFrame([demo_result]))
print("Integrated pipeline functional.")

## L. Pipeline Smoke Tests

In [ ]:
# Carbon calculator + pipeline validation tests
test_cases = [
    {
        "name": "Normal consumption",
        "previous_2": 92,
        "previous_1": 114,
        "month": 3
    },
    {
        "name": "Zero historical consumption",
        "previous_2": 0,
        "previous_1": 0,
        "month": 1
    },
    {
        "name": "High historical consumption",
        "previous_2": 800,
        "previous_1": 950,
        "month": 7
    }
]

test_results = []

for case in test_cases:
    result = forecast_next_month_co2(
        electricity_t_minus_2=case["previous_2"],
        electricity_t_minus_1=case["previous_1"],
        target_month_num=case["month"]
    )

    predicted_kwh = result["predicted_electricity_kwh"]
    predicted_co2 = result["predicted_co2_kg"]

    expected_co2 = round(
        predicted_kwh * ELECTRICITY_FACTOR["kg_co2_per_kwh"],
        3
    )

    assert np.isfinite(predicted_kwh)
    assert predicted_kwh >= 0
    assert np.isclose(predicted_co2, expected_co2)

    test_results.append({
        "Test": case["name"],
        "Predicted kWh": predicted_kwh,
        "Predicted CO2 (kg)": predicted_co2,
        "Status": "PASS"
    })

display(pd.DataFrame(test_results))
print(f"\n{len(test_results)}/{len(test_cases)} pipeline tests passed.")

## M. Model Metadata, Serialization, and Backend Package Export

In [ ]:
# Save the exact final model we just evaluated
joblib.dump(final_model, MODEL_PATH)

# Build complete metadata from actual configuration and evaluation
model_metadata = {
    "project": "CarbonIQ",
    "model_name": "Seasonal Gradient Boosting",
    "model_type": "GradientBoostingRegressor",
    "model_file": MODEL_PATH.name,

    "features": final_features,

    "target": "electricity_target",
    "target_unit": "kWh/month",

    "training": {
        "training_samples": int(len(X_train_final)),
        "testing_samples": int(len(X_test_final)),
        "training_cutoff": "Before November 2019",
        "testing_period": "November 2019 to February 2020",
        "dataset_period": "January 2018 to February 2020"
    },

    "evaluation": {
        metric: round(float(value), 4)
        for metric, value in final_metrics.items()
    },

    "model_parameters": final_model.get_params(),

    "emission_factor": ELECTRICITY_FACTOR,

    "limitations": [
        "Trained on historical household electricity observations.",
        "Evaluation uses a historical test period (Nov 2019–Feb 2020).",
        "Current-period forecasts need validation against newer data.",
        "The emission factor is configurable and separate from the ML model."
    ],

    "software": {
        "python_version": platform.python_version(),
        "scikit_learn_version": __import__('sklearn').__version__
    }
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(model_metadata, f, indent=4)

print("Model and metadata saved.")
print(f"Model: {MODEL_PATH}")
print(f"Metadata: {METADATA_PATH}")
print(f"\nEvaluation recorded:")
print(model_metadata["evaluation"])

In [ ]:
# Verify saved model loads and produces identical predictions
reloaded_model = joblib.load(MODEL_PATH)

sample_input = X_test_final.iloc[[0]]
original_pred = float(final_model.predict(sample_input)[0])
reloaded_pred = float(reloaded_model.predict(sample_input)[0])

print(f"Original prediction:  {original_pred:.6f}")
print(f"Reloaded prediction:  {reloaded_pred:.6f}")
assert abs(original_pred - reloaded_pred) < 1e-9
print("✅ Model reload verification: PASS")

# Verify metadata consistency
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    saved_metadata = json.load(f)

expected_metrics = {
    metric: round(float(value), 4)
    for metric, value in final_metrics.items()
}
assert saved_metadata["evaluation"] == expected_metrics
print("✅ Metadata consistency: PASS")

print(f"\nVerified metrics: {saved_metadata['evaluation']}")

In [ ]:
# Create FastAPI application source
api_code = r'''
from pathlib import Path
import json

import joblib
import pandas as pd
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

BASE_DIR = Path(__file__).resolve().parent

MODEL_PATH = BASE_DIR / "carboniq_seasonal_gb.joblib"
METADATA_PATH = BASE_DIR / "carboniq_model_metadata.json"

with open(METADATA_PATH, "r", encoding="utf-8") as file:
    metadata = json.load(file)

model = joblib.load(MODEL_PATH)

emission_factor = metadata["emission_factor"]
factor = emission_factor["kg_co2_per_kwh"]

app = FastAPI(
    title="CarbonIQ API",
    description="Electricity forecasting and carbon estimation",
    version="0.1.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173", "http://127.0.0.1:5173"],
    allow_credentials=True,
    allow_methods=["GET", "POST"],
    allow_headers=["*"]
)


class PredictionRequest(BaseModel):
    electricity_t_minus_2: float = Field(ge=0, allow_inf_nan=False)
    electricity_t_minus_1: float = Field(ge=0, allow_inf_nan=False)
    target_month_num: int = Field(ge=1, le=12)


@app.get("/health")
def health():
    return {
        "status": "ok",
        "model_loaded": model is not None,
        "model_name": metadata["model_name"]
    }


@app.post("/predict")
def predict(request: PredictionRequest):
    values = {
        "electricity_t_minus_2": request.electricity_t_minus_2,
        "electricity_t_minus_1": request.electricity_t_minus_1,
        "target_month_num": request.target_month_num
    }

    model_input = pd.DataFrame([
        {feature: values[feature] for feature in metadata["features"]}
    ])

    raw_prediction = float(model.predict(model_input)[0])
    predicted_kwh = round(max(0.0, raw_prediction), 2)
    predicted_co2 = round(predicted_kwh * factor, 3)

    return {
        "predicted_electricity_kwh": predicted_kwh,
        "predicted_co2_kg": predicted_co2,
        "emission_factor_kg_co2_per_kwh": factor,
        "factor_source": emission_factor["source"],
        "factor_version": emission_factor["database_version"],
        "reporting_period": emission_factor["reporting_period"],
        "gas": emission_factor["gas"],
        "model_name": metadata["model_name"],
        "model_metrics": metadata["evaluation"],
        "limitations": metadata["limitations"]
    }
'''

api_path = ARTIFACT_DIR / "app.py"
api_path.write_text(api_code.strip() + "\n", encoding="utf-8")

# Record dependencies for reproducible setup
requirements = f"""fastapi>=0.115,<1
uvicorn[standard]>=0.30,<1
scikit-learn=={__import__('sklearn').__version__}
numpy=={np.__version__}
pandas=={pd.__version__}
joblib=={joblib.__version__}
"""

requirements_path = ARTIFACT_DIR / "requirements.txt"
requirements_path.write_text(requirements, encoding="utf-8")

# Package model, metadata, API source and dependencies
with zipfile.ZipFile(BACKEND_ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in [
        "carboniq_seasonal_gb.joblib",
        "carboniq_model_metadata.json",
        "app.py",
        "requirements.txt"
    ]:
        archive.write(ARTIFACT_DIR / filename, arcname=filename)

print("CarbonIQ backend package created.")
print("\nFiles included:")
for f in ["carboniq_seasonal_gb.joblib", "carboniq_model_metadata.json", "app.py", "requirements.txt"]:
    print(f"  {f}")
print(f"\nPackage: {BACKEND_ZIP_PATH}")

In [ ]:
# Full API smoke tests: health, prediction, metadata consistency, invalid input rejection
import sys
import time
import subprocess
import requests
from pathlib import Path

LOG_PATH = ARTIFACT_DIR / "api_final_test.log"
BASE_URL = "http://127.0.0.1:8000"

# 1. Verify required backend files exist
required_files = ["app.py", "carboniq_seasonal_gb.joblib", "carboniq_model_metadata.json", "requirements.txt"]
for f in required_files:
    assert (ARTIFACT_DIR / f).exists(), f"Missing required file: {f}"

with open(ARTIFACT_DIR / "carboniq_model_metadata.json", "r", encoding="utf-8") as f:
    saved_metadata = json.load(f)

expected_metrics = {
    metric: round(float(value), 4)
    for metric, value in final_metrics.items()
}
assert saved_metadata["evaluation"] == expected_metrics

# 2. Start fresh API process
server = None
log_file = None

try:
    log_file = open(LOG_PATH, "w", encoding="utf-8")

    server = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"],
        cwd=str(ARTIFACT_DIR),
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

    health_response = None
    for _ in range(30):
        if server.poll() is not None:
            log_file.flush()
            raise RuntimeError("API process exited during startup\n" + LOG_PATH.read_text(encoding="utf-8"))
        try:
            r = requests.get(f"{BASE_URL}/health", timeout=1)
            if r.status_code == 200:
                health_response = r
                break
        except requests.RequestException:
            pass
        time.sleep(1)
    
    if health_response is None:
        log_file.flush()
        raise RuntimeError("API did not start within 30s\n" + LOG_PATH.read_text(encoding="utf-8"))

    # TEST 1: Health endpoint
    health = health_response.json()
    assert health["status"] == "ok"
    assert health["model_loaded"] is True
    assert health["model_name"] == "Seasonal Gradient Boosting"
    print("TEST 1: Health endpoint PASS")

    # TEST 2: Prediction, CO2 calculation, metadata consistency
    payload = {"electricity_t_minus_2": 92, "electricity_t_minus_1": 114, "target_month_num": 3}
    response = requests.post(f"{BASE_URL}/predict", json=payload, timeout=10)
    assert response.status_code == 200, f"Failed: {response.status_code} {response.text}"

    api_result = response.json()

    # Verify prediction matches the exact final trained model
    expected_input = pd.DataFrame([{feature: payload[feature] for feature in final_features}])
    raw = float(final_model.predict(expected_input)[0])
    expected_kwh = round(max(0.0, raw), 2)
    assert np.isclose(api_result["predicted_electricity_kwh"], expected_kwh, atol=1e-9)

    # Verify emission calculation
    expected_factor = ELECTRICITY_FACTOR["kg_co2_per_kwh"]
    assert np.isclose(api_result["emission_factor_kg_co2_per_kwh"], expected_factor)
    expected_co2 = round(expected_kwh * expected_factor, 3)
    assert np.isclose(api_result["predicted_co2_kg"], expected_co2, atol=1e-9)

    # Verify model metrics at same precision as saved metadata
    assert api_result["model_metrics"] == expected_metrics
    assert api_result["factor_source"] == saved_metadata["emission_factor"]["source"]

    print("TEST 2: Prediction and carbon calculation PASS")
    print("TEST 3: Model metadata consistency PASS")
    print("\nAPI prediction:")
    display(pd.DataFrame([api_result]))

    # TEST 4: Invalid input rejection
    invalid_response = requests.post(
        f"{BASE_URL}/predict",
        json={"electricity_t_minus_2": -10, "electricity_t_minus_1": 114, "target_month_num": 3},
        timeout=10
    )
    assert invalid_response.status_code == 422
    print("TEST 4: Invalid input rejection PASS")

    print("\nAll backend smoke tests passed: 4/4")

finally:
    if server is not None and server.poll() is None:
        server.terminate()
        try:
            server.wait(timeout=5)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait()
    if log_file is not None:
        log_file.close()
    print("Temporary test server stopped.")